# Extended DQN: two 75K T4 pilots

**This notebook launches V0 seed42 and V2 seed42 only. It does not launch the twelve-run matrix.**

Upload this notebook to https://colab.research.google.com, select Runtime > Change runtime type > T4 GPU, and run all cells. Upload the NEW extended source ZIP when prompted. The old 2K benchmark ZIP cannot run these experiments. The exact uncommitted integrated working revision is supplied as a SHA256-pinned archive; no unpublished remote branch is referenced.

Protocol: ls20-9607627b, ar25-0c556536, bp35-0a0ad940, round robin, 75K selected actions total =25K/game. Each pilot uses the same450K configuration: epsilon1 to0.1 over150K, replay20K, batch8, warmup1000, train every5, target250 updates, level reward, level resets after GAME_OVER, four-frame history only for V2. Expected updates **14,801 each**. Zero task reward does not invalidate a pilot.

Development only: ft09/vc33/cd82. sb26/sk48/su15 remain reserved, unconfirmed and absent from the bundle. No heldout evaluation is scheduled. Existing PPO finals trained503,808 actions on three games and use different rewards/action representations; this is not an algorithm-only fairness claim.


## CUDA compatibility repair - 24 October 2026

The earlier copy failed because AdaptiveAvgPool2d has no strict deterministic CUDA backward. The new source replaces it with mathematically equivalent non-overlapping16x16 average pooling for the fixed64x64 encoder. Strict determinism stays enabled and checkpoint parameter names/shapes stay compatible.

**Restart the Colab session, then run all cells and upload the NEW source ZIP requested below.** Old failed preflight folders are preserved. All test failures stop progression; never replace the test check with a warning. Training progress and error output now stream into notebook cells.


In [ ]:
import os,sys,subprocess,json,hashlib,zipfile,uuid,time,shutil,platform
from pathlib import Path
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
assert sys.version_info >= (3,12), "Use a Colab Python3.12+ runtime"
def run_checked(cmd):
    """Show subprocess errors in Colab and stop immediately on failure."""
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, start_new_session=True)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        code = process.wait()
    except BaseException:
        import signal
        try:
            os.killpg(process.pid, signal.SIGTERM)
            process.wait(timeout=10)
        except ProcessLookupError:
            pass
        raise
    if code:
        raise subprocess.CalledProcessError(code, cmd)
    return process


run_checked([sys.executable,"-m","pip","install","arc-agi==0.9.9","arcengine==0.9.3",
    "gymnasium==1.4.0","numpy==2.5.4","PyYAML==6.0.3","pytest>=8,<10"])
import torch
assert torch.cuda.is_available(), "Select T4 GPU and reconnect"
assert "T4" in torch.cuda.get_device_name(0), torch.cuda.get_device_name(0)
assert (2,8) <= tuple(map(int,torch.__version__.split('+')[0].split('.')[:2])) < (3,0)
run_checked(["nvidia-smi"])
print("Python:",sys.version,"Torch:",torch.__version__,"CUDA:",torch.version.cuda)
print(Path('/proc/meminfo').read_text().splitlines()[:3])


In [ ]:
from google.colab import files
uploaded=files.upload()  # Upload only dqn_extended_source_20261024T210856472779Z.zip
assert len(uploaded)==1
name,raw=next(iter(uploaded.items()))
EXPECTED_BUNDLE_SHA256="91f2735dc3ef874fecb87d79f848a11897628db9705f3f4d07ae5412eed79b32"
assert hashlib.sha256(raw).hexdigest()==EXPECTED_BUNDLE_SHA256,"Wrong or changed source ZIP"
ROOT=Path('/content')/('arc_dqn_extended_'+uuid.uuid4().hex[:12])
ROOT.mkdir(exist_ok=False)
with zipfile.ZipFile(__import__('io').BytesIO(raw)) as archive:
    for member in archive.namelist():
        assert (ROOT/member).resolve().is_relative_to(ROOT.resolve())
    archive.extractall(ROOT)
manifest=json.loads((ROOT/'EXTENDED_SOURCE_MANIFEST.json').read_text())
for path,expected in manifest['sha256'].items():
    assert hashlib.sha256((ROOT/path).read_bytes()).hexdigest()==expected,path
os.chdir(ROOT)
sys.path.insert(0,str(ROOT))
print('Base commit:',manifest['base_commit'],'Exact bundle:',EXPECTED_BUNDLE_SHA256)


## Persistence and preflight

Mount Drive before training. Results go into a new timestamped directory in MyDrive/ARC_DQN_extended. Full learner state is written and checksummed every25K actions; only the latest two complete continuation generations are retained in each new run. Frozen75K model snapshots remain separate. Snapshot/persistence time is included in reported end-to-end throughput.

Allow roughly8GiB free RAM. Full replay payload at20K is about246MB (one frame) or738MB (four frames), before Python objects and temporary copies. Reserve about25GB persistent storage for the eventual complete matrix unless pilot compression proves a lower need. Drive quota is not the VM disk space shown below. Colab availability/session limits fluctuate; no uninterrupted session is assumed.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PERSIST=Path('/content/drive/MyDrive/ARC_DQN_extended')
PERSIST.mkdir(parents=True,exist_ok=True)
LOCAL=ROOT/'runs'
LOCAL.mkdir(exist_ok=True)
run_checked([sys.executable,'-m','Experiments.prepare_extended_assets','--include-development'])
run_checked([sys.executable,'-m','pytest','-q','tests/test_dqn_extended.py',
    'tests/test_dqn_devices.py','tests/test_dqn_shared_training.py'])
print('VM disk:',shutil.disk_usage('/content'))
with (ROOT/'pip_freeze.txt').open('w') as f:
    subprocess.run([sys.executable,'-m','pip','freeze'],stdout=f,check=True)


## Run the two pilots

The runner first performs a real three-game CUDA reconstruction test. Any response/parameter divergence stops progression. The pilots then run serially. Progress and any failures are saved in console files; the full matrix is absent from this executable notebook. Initial planning allowance: roughly15-30 minutes for both pilots, subject to actual history/game/Drive costs.


In [ ]:
cmd=[sys.executable,'-m','Experiments.run_dqn_extended','--phase','pilots',
     '--output-root',str(LOCAL),'--persistent-root',str(PERSIST),'--device','cuda']
before=set(LOCAL.glob('pilots_*'))
run_checked(cmd)
batches=sorted(set(LOCAL.glob('pilots_*'))-before)
assert len(batches)==1
BATCH=batches[0]
print('Pilot batch:',BATCH)
print((BATCH/'matrix_cost_estimate.json').read_text())
for p in BATCH.glob('*/pilot_validation.json'):
    v=json.loads(p.read_text())
    print(v['variant'],v['required_actions'],v['expected_updates'],v['per_game_actions'],
          'actions/s including persistence:',v['actions_per_second_including_persistence'])


## Common Task3 development evaluation

Evaluate only the declared75K snapshots and matched random reference: three exposed games x evaluation seeds0/1/2. Official local score, level/game completion, action distributions and failures are recorded. No novelty is added to evaluation rewards. Repeated evaluation seeds do not count as additional independent training seeds. Pilot seed42 alone cannot estimate between-training-seed variability.


In [ ]:
DEV=BATCH/'development_evaluation'
run_checked([sys.executable,'-m','Experiments.evaluate_dqn_extended',str(BATCH),
    '--steps','75000','--out',str(DEV),'--device','cuda'])
from DQN.extended_train import publish
# Keep source/dependency evidence in the persistent batch.
shutil.copy2(ROOT/'EXTENDED_SOURCE_MANIFEST.json',BATCH)
shutil.copy2(ROOT/'pip_freeze.txt',BATCH)
publish(BATCH,PERSIST)
print((DEV/'evaluation_validation.json').read_text())
print((DEV/'training_seed_aggregate.json').read_text())


In [ ]:
# Compact handoff ZIP. Full replay payloads stay safely on Drive.
EVIDENCE=ROOT/(BATCH.name+'_evidence.zip')
with zipfile.ZipFile(EVIDENCE,'x',compression=zipfile.ZIP_DEFLATED) as archive:
    for p in BATCH.rglob('*'):
        if not p.is_file() or 'resume' in p.parts or 'reconstruction' in p.parts:
            continue
        archive.write(p,p.relative_to(BATCH).as_posix())
print('Persistent complete checkpoints:',PERSIST/BATCH.name)
print('Send this evidence ZIP back for pilot interpretation:',EVIDENCE)
files.download(str(EVIDENCE))


## If interrupted

The latest complete Drive generation has `state.pt`, `replay.npz`, and `COMPLETE.json`. Upload the SAME source archive, recreate the recorded Python/PyTorch/package versions, mount Drive, copy one complete generation to VM disk, then use a NEW output folder. Resume reconstructs the exact seeded action/reset transcript and checks every response before restoring replay/optimizer/RNG/novelty/history. A version/source/trajectory mismatch fails; do not discard that check. This can take extra CPU time proportional to all past actions.

Example command (edit paths; text only, not automatically run):

`python -m DQN.extended_train --config Configs/dqn_extended_v1.json --variant v2 --seed 42 --device cuda --resume /content/copied_generation --run-dir /content/new_continuation_run --stop-at 75000 --persistent-root /content/drive/MyDrive/ARC_DQN_extended/recovered`

Restoration copies the compressed committed action-log prefix into the new run, then appends only new global action indices. Unsaved old tails are excluded. A continued pilot is not automatically accepted by the fresh-pilot matrix gate because its throughput spans sessions; request an audit of the combined timing evidence or restart the short pilot. Full450K continuations preserve parent linkage and complete action histories.

## After pilot review only

The matrix runner requires both valid75K CUDA pilot results plus successful shared development evaluation under matching source/protocol/game hashes. It defaults to fresh full runs, preserving pilots separately. To reuse validated prefixes deliberately, use the continuation command with stop-at450000 and the unchanged protocol; this is not done automatically.

Separate future command (NOT an executable notebook cell):

`python -m Experiments.run_dqn_extended --phase matrix --validated-pilots /content/pilots_batch --output-root /content/full_matrix --persistent-root /content/drive/MyDrive/ARC_DQN_extended --device cuda`

Zero completed levels does not change the ablation hypotheses. Verify infrastructure and costs, then decide the full budget. The report must distinguish old3K/native experiments, new shared three-game450K runs and PPO503,808-action finals. Heldout confirmation remains outstanding.
